# TRELLIS.2 — Drive Batch Image → 3D

## Use
1. Put images in **`MyDrive/Shared/Trellis/input`**.
2. In Colab select a **GPU runtime**.
3. Click **Runtime → Run all**.

The notebook installs/loads TRELLIS once, processes every PNG/JPG/JPEG/WEBP in the input folder, saves each model as a `.glb`, and stops when the folder is empty.

### Drive folders
- Input: `MyDrive/Shared/Trellis/input`
- Output: `MyDrive/Shared/Trellis/output`
- Finished inputs: `MyDrive/Shared/Trellis/done`
- Failed inputs: `MyDrive/Shared/Trellis/failed`
- Build cache: `MyDrive/Shared/Trellis/cache`

> If `HF_TOKEN` is stored in Colab Secrets there are no token prompts. Otherwise the notebook asks for the Hugging Face read token once.


In [ ]:
from google.colab import drive, userdata
from pathlib import Path
import getpass, os, shutil, subprocess

drive.mount('/content/drive', force_remount=False)
ROOT = Path('/content/drive/MyDrive/Shared/Trellis')
INPUT_DIR = ROOT / 'input'
OUTPUT_DIR = ROOT / 'output'
DONE_DIR = ROOT / 'done'
FAILED_DIR = ROOT / 'failed'
CACHE_DIR = ROOT / 'cache'
for folder in (INPUT_DIR, OUTPUT_DIR, DONE_DIR, FAILED_DIR, CACHE_DIR):
    folder.mkdir(parents=True, exist_ok=True)

print('Input :', INPUT_DIR)
print('Output:', OUTPUT_DIR)

REPO = Path('/content/My-works')
if REPO.exists():
    shutil.rmtree(REPO)
subprocess.run([
    'git', 'clone', '--depth', '1',
    'https://github.com/Logan17de/My-works.git', str(REPO)
], check=True)
TOOLS_3D = REPO / 'ai-3d-animation-engines' / '3d-engine'

os.environ['ENGINE_CACHE_ROOT'] = str(CACHE_DIR)
subprocess.run(['bash', str(TOOLS_3D / 'install_3d.sh')], check=True)

try:
    HF_TOKEN = userdata.get('HF_TOKEN')
except Exception:
    HF_TOKEN = None
if not HF_TOKEN:
    HF_TOKEN = getpass.getpass('Hugging Face READ token: ').strip()
if not HF_TOKEN:
    raise RuntimeError('HF_TOKEN is required.')

env = os.environ.copy()
env['HF_TOKEN'] = HF_TOKEN
env['HF_HOME'] = '/content/huggingface'
env['HF_XET_HIGH_PERFORMANCE'] = '1'
env['PYTHONUNBUFFERED'] = '1'

subprocess.run([
    '/opt/conda/bin/conda', 'run', '--no-capture-output', '-n', 'trellis2',
    'python', str(TOOLS_3D / 'prepare_hf_models.py')
], cwd='/content/TRELLIS.2', env=env, check=True)

subprocess.run([
    '/opt/conda/bin/conda', 'run', '--no-capture-output', '-n', 'trellis2',
    'python', str(TOOLS_3D / 'batch_drive_trellis2.py')
], cwd='/content/TRELLIS.2', env=env, check=True)
